# 06 — Packet Analysis with Python

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Network Security  
**Tools:** Scapy, Python 3

---

## Objectives
- Understand packet structure (Ethernet, IP, TCP/UDP)
- Parse and inspect packets using Scapy
- Detect suspicious traffic patterns
- Summarize captured traffic statistics

## 1. Setup & Imports

In [ ]:
# Install scapy if not already installed
# !pip install scapy

from scapy.all import rdpcap, IP, TCP, UDP, ICMP
from collections import Counter
import json

## 2. Packet Structure Overview

A typical network packet has these layers:

```
[ Ethernet Frame ]
  └── [ IP Header ]  — src/dst IP, TTL, protocol
        └── [ TCP/UDP Header ] — src/dst port, flags
              └── [ Payload / Application Data ]
```

| Layer | Protocol | Key Fields |
|-------|----------|------------|
| 2 | Ethernet | src/dst MAC |
| 3 | IP | src/dst IP, TTL |
| 4 | TCP/UDP | src/dst port, flags |
| 7 | HTTP/DNS | Application payload |

## 3. Reading a PCAP File

In [ ]:
# Load a .pcap file (replace with your own capture file)
# packets = rdpcap('sample.pcap')

# --- Simulated packets for demo ---
from scapy.all import Ether, IP, TCP, UDP, ICMP

packets = [
    Ether() / IP(src='192.168.1.10', dst='8.8.8.8', ttl=64) / TCP(sport=54321, dport=80, flags='S'),
    Ether() / IP(src='192.168.1.10', dst='8.8.8.8', ttl=64) / TCP(sport=54321, dport=443, flags='S'),
    Ether() / IP(src='10.0.0.5',    dst='192.168.1.10', ttl=128) / UDP(sport=53, dport=1024),
    Ether() / IP(src='10.0.0.1',    dst='192.168.1.10', ttl=64) / ICMP(),
    Ether() / IP(src='203.0.113.5', dst='192.168.1.10', ttl=50) / TCP(sport=4444, dport=22, flags='S'),
]

print(f'Total packets loaded: {len(packets)}')

## 4. Inspecting Individual Packets

In [ ]:
for i, pkt in enumerate(packets):
    if IP in pkt:
        proto = 'TCP' if TCP in pkt else 'UDP' if UDP in pkt else 'ICMP'
        src = pkt[IP].src
        dst = pkt[IP].dst
        ttl = pkt[IP].ttl
        print(f'Packet {i+1}: {proto}  {src} -> {dst}  TTL={ttl}')

## 5. Traffic Statistics

In [ ]:
protocols = []
src_ips   = []
dst_ports = []

for pkt in packets:
    if IP in pkt:
        src_ips.append(pkt[IP].src)
        if TCP in pkt:
            protocols.append('TCP')
            dst_ports.append(pkt[TCP].dport)
        elif UDP in pkt:
            protocols.append('UDP')
            dst_ports.append(pkt[UDP].dport)
        elif ICMP in pkt:
            protocols.append('ICMP')

print('Protocol counts:', dict(Counter(protocols)))
print('Top source IPs: ', Counter(src_ips).most_common(5))
print('Top dest ports:', Counter(dst_ports).most_common(5))

## 6. Detecting Suspicious Traffic

Common indicators of suspicious activity:
- **SYN flood** — many SYN packets from one source with no ACK
- **Port scanning** — one source hitting many ports
- **Unusual ports** — traffic on non-standard ports (e.g. 4444, 1337)
- **Low TTL** — may indicate crafted/spoofed packets

In [ ]:
SUSPICIOUS_PORTS = {4444, 1337, 31337, 6666, 9999}
alerts = []

for i, pkt in enumerate(packets):
    if IP in pkt:
        # Check suspicious destination ports
        if TCP in pkt and pkt[TCP].dport in SUSPICIOUS_PORTS:
            alerts.append(f'[!] Packet {i+1}: Suspicious port {pkt[TCP].dport} from {pkt[IP].src}')
        # Check very low TTL (possible spoofing)
        if pkt[IP].ttl < 10:
            alerts.append(f'[!] Packet {i+1}: Low TTL ({pkt[IP].ttl}) from {pkt[IP].src}')

if alerts:
    print('=== ALERTS ===')
    for a in alerts:
        print(a)
else:
    print('No suspicious traffic detected.')

## 7. Export Summary to JSON

In [ ]:
summary = {
    'total_packets': len(packets),
    'protocol_counts': dict(Counter(protocols)),
    'top_source_ips': dict(Counter(src_ips).most_common(5)),
    'top_dest_ports': dict(Counter(dst_ports).most_common(5)),
    'alerts': alerts
}

print(json.dumps(summary, indent=2))

## 8. Key Takeaways

- Scapy allows deep packet inspection at every layer
- Protocol distribution gives a quick health check of network traffic
- Port and IP frequency analysis reveals scanning or flood attacks
- Always correlate alerts with context before escalating

---
*Next: 07 — Log Analysis & SIEM Basics*